# CNN–VGG16 — Modelo A (Train)

Variación directa de `VGG16_Train.ipynb`. La sección convolucional no se modifica.


In [ ]:
# EJECUTA ESTA CELDA PRIMERO. La primera vez reiniciará el kernel automáticamente.
import os, signal, subprocess, sys
from pathlib import Path
marcador = Path("/tmp/vgg16_dependencias_compatibles")
if not marcador.exists():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "protobuf==6.31.1", "tensorflow-datasets>=4.9,<5.0",
        "scikit-learn>=1.6,<2.0", "seaborn==0.13.2", "pydot", "graphviz",
    ])
    marcador.touch()
    print("Dependencias instaladas. El kernel se reiniciará; después vuelve a ejecutar esta celda.")
    os.kill(os.getpid(), signal.SIGKILL)
print("Dependencias compatibles listas. Continúa con la siguiente celda.")


In [ ]:
import hashlib, importlib.metadata, json, os, platform, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

SEMILLA = 42
TAMANIO_IMAGEN = (224, 224)
BATCH_SIZE = 16
EPOCAS = 30
CLASES = ["dandelion", "daisy", "tulips", "sunflowers", "roses"]
# ADAPTADO: después de reiniciar el kernel, Colab vuelve a /content.
# Si Drive está montado, guardar siempre dentro de la carpeta del proyecto.
RAIZ_DRIVE = Path("/content/drive/MyDrive/vgg16_tf_flowers")
RAIZ = RAIZ_DRIVE if RAIZ_DRIVE.exists() else Path.cwd()
for carpeta in ["data", "weights", "results", "figures"]:
    (RAIZ / carpeta).mkdir(exist_ok=True)
print("Los artefactos se guardarán en:", RAIZ.resolve())

os.environ["PYTHONHASHSEED"] = str(SEMILLA)
random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.keras.utils.set_random_seed(SEMILLA)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("TensorFlow:", tf.__version__)
print("TFDS:", importlib.metadata.version("tensorflow-datasets"))
print("Python:", platform.python_version())
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
# 1. Cargar las imágenes: ADAPTADO de directorios a tf_flowers (5 clases)
dataset_base, info = tfds.load(
    "tf_flowers:3.0.1", split="train", as_supervised=True,
    with_info=True, shuffle_files=False
)
assert info.features["label"].num_classes == 5
CLASES = list(info.features["label"].names)
TOTAL = info.splits["train"].num_examples
etiquetas = np.fromiter((int(y) for _, y in tfds.as_numpy(dataset_base)), dtype=np.int64, count=TOTAL)
indices = np.arange(TOTAL, dtype=np.int64)

# Partición estratificada 70/15/15; la misma semilla produce el mismo manifiesto.
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, etiquetas, test_size=0.30, random_state=SEMILLA, stratify=etiquetas
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=SEMILLA, stratify=y_temp
)
asignacion = np.empty(TOTAL, dtype=object)
asignacion[idx_train], asignacion[idx_val], asignacion[idx_test] = "train", "validation", "test"
manifiesto = pd.DataFrame({"index": indices, "label": etiquetas, "split": asignacion})
ruta_manifiesto = RAIZ / "data" / "split_manifest.csv"
if ruta_manifiesto.exists():
    anterior = pd.read_csv(ruta_manifiesto)
    pd.testing.assert_frame_equal(anterior, manifiesto, check_dtype=False)
else:
    manifiesto.to_csv(ruta_manifiesto, index=False)
assert manifiesto["index"].is_unique and len(manifiesto) == TOTAL
assert set(manifiesto["split"]) == {"train", "validation", "test"}

def seleccionar(indices_split):
    llaves = tf.constant(np.asarray(indices_split), dtype=tf.int64)
    tabla = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(llaves, tf.ones_like(llaves, dtype=tf.int32)),
        default_value=0,
    )
    ds = dataset_base.enumerate()
    ds = ds.filter(lambda i, elemento: tabla.lookup(i) > 0)
    return ds.map(lambda i, elemento: elemento, num_parallel_calls=tf.data.AUTOTUNE)

aumento = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEMILLA),
    tf.keras.layers.RandomRotation(0.08, seed=SEMILLA),
    tf.keras.layers.RandomZoom(0.10, seed=SEMILLA),
], name="aumento_solo_entrenamiento")

def preparar(ds, entrenando=False):
    ds = ds.map(
        lambda x, y: (tf.image.resize(tf.image.convert_image_dtype(x, tf.float32), TAMANIO_IMAGEN), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    if entrenando:
        ds = ds.shuffle(1024, seed=SEMILLA, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

entrenamiento = preparar(seleccionar(idx_train), entrenando=True)
validacion = preparar(seleccionar(idx_val))
prueba = preparar(seleccionar(idx_test))

# Pruebas rápidas de integridad.
x_lote, y_lote = next(iter(validacion))
assert x_lote.shape[1:] == (224, 224, 3)
assert 0 <= int(tf.reduce_min(y_lote)) and int(tf.reduce_max(y_lote)) < 5
assert len(set(idx_train) & set(idx_val)) == len(set(idx_train) & set(idx_test)) == 0
print(info)
print(manifiesto.groupby(["split", "label"]).size().unstack(fill_value=0))


In [ ]:
# ADAPTADO: mosaico e histograma del conjunto de datos
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (imagen, etiqueta) in zip(axes.flat, tfds.as_numpy(dataset_base.take(10))):
    ax.imshow(imagen)
    ax.set_title(CLASES[int(etiqueta)])
    ax.axis("off")
fig.tight_layout()
fig.savefig(RAIZ / "figures" / "mosaico_dataset.png", dpi=180, bbox_inches="tight")
plt.show()

conteos = manifiesto.groupby(["split", "label"]).size().reset_index(name="imagenes")
conteos["clase"] = conteos["label"].map(dict(enumerate(CLASES)))
plt.figure(figsize=(11, 5))
sns.barplot(data=conteos, x="clase", y="imagenes", hue="split")
plt.title("Distribución estratificada de tf_flowers")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(RAIZ / "figures" / "histograma_clases_splits.png", dpi=180)
plt.show()


In [ ]:
# 2. Cargar VGG16 con pesos entrenados en ImageNet
vgg16 = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# 3. Congelar los pesos VGG16: sección convolucional SIN MODIFICACIONES
vgg16.trainable = False

# 4. Agregar el clasificador del AO_A
entrada = tf.keras.Input(shape=(224, 224, 3), name="imagen_RGB")
x = aumento(entrada)
x = preprocess_input(x * 255.0)
x = vgg16(x, training=False)
x = GlobalAveragePooling2D(name="promedio_global")(x)
salida = Dense(5, activation="softmax", name="salida_5_clases")(x)
modelo = Model(entrada, salida, name="VGG16_Modelo_A")
assert not vgg16.trainable and all(not capa.trainable for capa in vgg16.layers)
modelo.summary()

with open(RAIZ / "results" / "modelo_A_summary.txt", "w") as archivo:
    modelo.summary(print_fn=lambda linea: archivo.write(linea + "\n"))

# Diagrama pedagógico parecido al proporcionado: backbone fijo y cabezal variable.
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")
bloques = [
    ("Entrada\n224×224×3", "#d9edf7", 1.3),
    ("VGG16 convolucional\nCONGELADA\n14,714,688 parámetros", "#7fcdbb", 3.2),
    ("GlobalAveragePooling2D\n7×7×512 → 512", "#c7e9c0", 2.4),
] + [('Dense 5\nSoftmax', '#fdae6b', 1.7)]
x0 = 0
for texto, color, ancho in bloques:
    ax.add_patch(plt.Rectangle((x0, 0.8), ancho, 1.5, facecolor=color, edgecolor="black"))
    ax.text(x0 + ancho/2, 1.55, texto, ha="center", va="center", fontsize=9)
    x0 += ancho + 0.45
    if x0 < sum(b[2] + 0.45 for b in bloques):
        ax.annotate("", xy=(x0, 1.55), xytext=(x0-0.45, 1.55), arrowprops={"arrowstyle":"->"})
ax.set_xlim(-0.2, x0)
ax.set_ylim(0.4, 2.7)
ax.set_title("Modelo A: solo cambia el cabezal clasificador", fontweight="bold")
fig.tight_layout()
fig.savefig(RAIZ / "figures" / "arquitectura_modelo_A.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# 5. Compilar
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
ruta_pesos = RAIZ / "weights" / "modelo_A_best.weights.h5"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ruta_pesos, monitor="val_loss", save_best_only=True, save_weights_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-6),
]

# 6. Entrenar únicamente el nuevo clasificador
inicio = time.time()
historia = modelo.fit(entrenamiento, validation_data=validacion, epochs=EPOCAS, callbacks=callbacks)
duracion = time.time() - inicio
assert all(not capa.trainable for capa in vgg16.layers)

# 7. Guardar modelo, pesos, historial y configuración
modelo.load_weights(ruta_pesos)
ruta_modelo = RAIZ / "weights" / "modelo_A.keras"
modelo.save(ruta_modelo)
historial = pd.DataFrame(historia.history)
historial.to_csv(RAIZ / "results" / "modelo_A_historial.csv", index=False)
metadata = {
    "modelo": "A", "semilla": SEMILLA, "dataset": "tf_flowers:3.0.1",
    "clases": CLASES, "mejor_epoca": int(historial["val_loss"].idxmin() + 1),
    "duracion_segundos": duracion,
    "parametros_totales": int(modelo.count_params()),
    "parametros_entrenables": int(sum(np.prod(v.shape) for v in modelo.trainable_weights)),
}
for ruta in [ruta_pesos, ruta_modelo]:
    metadata[ruta.name + "_sha256"] = hashlib.sha256(ruta.read_bytes()).hexdigest()
(RAIZ / "results" / "modelo_A_metadata.json").write_text(json.dumps(metadata, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historial["loss"], label="entrenamiento")
axes[0].plot(historial["val_loss"], label="validación")
axes[0].set(title="Loss Modelo A", xlabel="Época", ylabel="Loss")
axes[1].plot(historial["accuracy"], label="entrenamiento")
axes[1].plot(historial["val_accuracy"], label="validación")
axes[1].set(title="Accuracy Modelo A", xlabel="Época", ylabel="Accuracy")
for ax in axes: ax.legend(); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(RAIZ / "figures" / "curvas_modelo_A.png", dpi=180)
plt.show()

# Verificación de recarga y predicción
recargado = tf.keras.models.load_model(ruta_modelo)
assert recargado.predict(x_lote[:1], verbose=0).shape == (1, 5)
print(json.dumps(metadata, indent=2))
